# 替え歌動画をColabで生成する

XF MIDI(+編集済みの替え歌JSON)から、NEUTRINOの歌唱合成つき替え歌動画を作ります。

## 事前準備(初回のみ)

1. [NEUTRINO公式サイト](https://studio-neutrino.com/)から **Google Colab版(またはLinux版)** をダウンロードし、
   自分のGoogle Driveの `Colab Notebooks/NEUTRINO` に展開して置く([公式手順](https://studio-neutrino.com/561/))
2. ランタイム → ランタイムのタイプを変更 → **GPU** を選択(Diffusion系モデルはGPU前提)

## 使い方

上から順にセルを実行。途中で `song.mid`(XF形式)をアップロードします。
替え歌の中身を自分で調整したい場合は、[soramimic](https://soramimic.com) の
「MIDIから取り込み → 変換 → 編集ツール → 書き出し」で作ったJSONも一緒にアップロードしてください。

In [ ]:
# @title 1. セットアップ(3分ほど)
import os
import subprocess
from getpass import getpass

!apt-get -qq update && apt-get -qq install -y ffmpeg fluidsynth fluid-soundfont-gm fonts-noto-cjk nodejs npm > /dev/null

if not os.path.exists("soramimic-video"):
    # privateリポジトリの間はGitHubトークン(repo読み取り)が必要。公開後は空Enterでよい
    token = getpass("GitHub token(公開リポジトリなら空Enter): ").strip()
    if token:
        subprocess.run([
            "git", "config", "--global",
            f"url.https://{token}@github.com/.insteadOf", "https://github.com/",
        ], check=True)
    subprocess.run(["git", "clone", "--recursive",
                    "https://github.com/soramimic/soramimic-video.git"], check=True)

!cd soramimic-video && pip -q install -e .
!cd soramimic-video/bridge && npm ci --silent
print("セットアップ完了")

In [ ]:
# @title 2. Google Driveを接続(NEUTRINOの場所)
import os
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

# NEUTRINOを置いた場所(事前準備1)に合わせて変更可
NEUTRINO_ROOT = "/content/drive/MyDrive/Colab Notebooks/NEUTRINO"  # @param {type:"string"}

assert (Path(NEUTRINO_ROOT) / "bin").exists(), (
    f"{NEUTRINO_ROOT}/bin がありません。事前準備1を確認してください")
os.environ["NEUTRINO_ROOT"] = NEUTRINO_ROOT
!chmod -R u+x "$NEUTRINO_ROOT/bin"
print("モデル一覧:", sorted(p.name for p in (Path(NEUTRINO_ROOT) / "model").iterdir() if p.is_dir()))

In [ ]:
# @title 3. 入力ファイルをアップロード
# song.mid(XF形式・必須)、lyrics.txt(元歌詞・任意)、
# soramimic編集ツールで書き出したJSON(任意)をまとめて選択
from google.colab import files

uploaded = files.upload()
midi_path = next((n for n in uploaded if n.endswith((".mid", ".midi"))), None)
lyrics_path = next((n for n in uploaded if n.endswith(".txt")), None)
editor_json = next((n for n in uploaded if n.endswith(".json")), None)
assert midi_path, "XF MIDI(.mid)をアップロードしてください"
print("MIDI:", midi_path, "/ 元歌詞:", lyrics_path, "/ 編集JSON:", editor_json)

In [ ]:
# @title 4. 変換〜動画生成
import subprocess

WORDLIST = "stations"  # @param ["stations", "baseball", "football", "nations", "pokemon", "physicist", "sekitsui"]
WHERE = "status=current"  # @param {type:"string"}
MODEL = "MERROW"  # @param {type:"string"}

def run(*args):
    print("$", " ".join(args))
    subprocess.run(args, check=True, cwd="soramimic-video")

project = "work/song"
analyze = ["soramimic-video", "analyze", "--midi", f"../{midi_path}", "--project", project]
if lyrics_path:
    analyze += ["--lyrics", f"../{lyrics_path}"]
run(*analyze)
run("soramimic-video", "convert", "--project", project,
    "--wordlist", WORDLIST, *(["--where", WHERE] if WHERE else []))
if editor_json:
    run("soramimic-video", "import-editor", "--project", project, "--file", f"../{editor_json}")
run("soramimic-video", "synthesize", "--project", project, "--model", MODEL)
run("soramimic-video", "mix", "--project", project,
    "--soundfont", "/usr/share/sounds/sf2/FluidR3_GM.sf2")
run("soramimic-video", "video", "--project", project, "--font", "Noto Sans CJK JP")
print("完成: soramimic-video/work/song/video/out.mp4")

In [ ]:
# @title 5. 動画をダウンロード
from google.colab import files

files.download("soramimic-video/work/song/video/out.mp4")
# 画像クレジット(公開時に必要)も一緒にどうぞ
import os

if os.path.exists("soramimic-video/work/song/video/credits.md"):
    files.download("soramimic-video/work/song/video/credits.md")

## トラブルシュート

- **NEUTRINOのコマンドが失敗する**: バージョンによりCLIが異なります。
  `soramimic-video/work/song/neutrino/commands.json` をお使いのNEUTRINOの `Run.sh` に合わせて編集して、セル4を再実行してください
- **日本語字幕が豆腐になる**: セル4の `--font` をインストール済みフォント名に変更
  (`!fc-list :lang=ja` で確認)
- **replace 前の歌詞で歌ってしまう**: 編集JSONの取り込み(セル4のimport-editor)が行われているかログを確認